# RoleColor AI: Professional Persona & Summary Generator

This notebook implements an NLP pipeline to analyze professional resumes through the lens of four behavioral archetypes: **Builder**, **Enabler**, **Thriver**, and **Supportee**. 

### Key Features:
* **Role Scoring:** Uses TF-IDF vectorization and Cosine Similarity to determine your dominant "RoleColor" based on professional linguistic patterns.
* **Achievement Extraction:** Leverages spaCy NER and rule-based ranking to identify high-impact achievements while ensuring personal data privacy.
* **LLM Rewriting:** Utilizes the `flan-t5-large` model to generate a tailored, 4-line professional summary that aligns your experience with your dominant persona.

---

In [1]:
import re
import spacy
import torch
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Part 1: Rolo Color Key Words Map

In [2]:
# Configuration
KEYWORD_MAP = {
    "Builder": ["innovation", "vision", "strategy", "architecture", "innovate", "strategic", "visionary", "pioneer", "create", "develop", "design", "lead", "initiate", "drive"],
    "Enabler": ["collaboration", "coordinate", "cross-functional", "bridge", "connect", "facilitate", "support", "assist", "help", "enable", "liaise", "teamwork"],
    "Thriver": ["fast-paced", "deadline", "adapt", "pressure", "agile", "resilient", "quick", "dynamic", "thrive", "flexible", "rapid", "urgent"],
    "Supportee": ["reliability", "consistent", "stability", "maintenance", "documentation", "dependable", "stable", "reliable", "support", "maintain", "document", "consistent"]
}

ROLE_DESCRIPTIONS = {
    "Builder": "Drives innovation, creates vision, defines strategy, and builds architecture. Focuses on long‑term planning and pioneering new solutions.",
    "Enabler": "Connects people, facilitates collaboration, coordinates cross‑functional efforts, and bridges gaps. Supports teammates and executes plans.",
    "Thriver": "Thrives under pressure, adapts quickly to change, meets tight deadlines, and remains resilient in fast‑paced environments.",
    "Supportee": "Ensures reliability, maintains consistency, documents processes, and provides stable support. Focuses on dependability and quality."
}

# Part 2: Resume RoleColor Scoring

In [3]:
def build_role_corpus():
    """Create weighted documents for each RoleColor."""
    corpus = []
    for role in KEYWORD_MAP:
        # Repetition gives keywords more weight relative to the description
        text = f"{' '.join(KEYWORD_MAP[role] * 3)} {ROLE_DESCRIPTIONS[role]}"
        corpus.append(text)
    return corpus, list(KEYWORD_MAP.keys())

In [4]:
def score_resume(resume_text):
    """Compute normalized cosine similarities using TF-IDF."""
    role_corpus, role_names = build_role_corpus()
    documents = role_corpus + [resume_text]
    
    vectorizer = TfidfVectorizer(stop_words='english', lowercase=True)
    tfidf_matrix = vectorizer.fit_transform(documents)
    
    role_vectors = tfidf_matrix[:-1]
    resume_vector = tfidf_matrix[-1]
    
    similarities = cosine_similarity(resume_vector, role_vectors).flatten()
    similarities = np.maximum(similarities, 0) # Remove negative noise
    
    total = similarities.sum()
    if total == 0:
        return {role: 0.25 for role in role_names}
    
    scores = similarities / total
    return dict(zip(role_names, scores))

In [5]:
with open('resume.txt', 'r', encoding='utf-8') as f:
    text = f.read()
    
    # Process
    scores = score_resume(text)
    print(scores)
    dominant = max(scores, key=scores.get)
    print(f"\nDominant Role: {dominant} ({scores[dominant]:.2%})")

{'Builder': 0.22373474603797233, 'Enabler': 0.48628154418182407, 'Thriver': 0.056910704877907774, 'Supportee': 0.23307300490229588}

Dominant Role: Enabler (48.63%)


# Part 3: Resume Summary Rewrite

In [6]:
# Cache for model/tokenizer
_model, _tokenizer = None, None

def load_resources():
    global _model, _tokenizer
    nlp = spacy.load("en_core_web_sm") if spacy.util.is_package("en_core_web_sm") else spacy.cli.download("en_core_web_sm") or spacy.load("en_core_web_sm")
    
    if _model is None:
        _tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-large")
        _model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-large")
        if torch.cuda.is_available():
            _model = _model.to("cuda")
    return nlp, _model, _tokenizer

In [7]:
def extract_metadata(text, nlp):
    """
    Extracts job title and top achievements while filtering out 
    personal names (NER) from the text.
    """
    # 1. Clean the text to remove person names using spaCy NER
    doc_clean = nlp(text)
    # Create a list of names to filter out
    person_names = [ent.text for ent in doc_clean.ents if ent.label_ == "PERSON"]
    
    cleaned_text = text
    for name in person_names:
        # Use regex to replace the specific name with an empty string (case sensitive)
        cleaned_text = re.sub(re.escape(name), "", cleaned_text)

    # 2. Extract Job Title from cleaned text
    title = "Professional"
    # Looking for titles like 'Machine Learning Engineer' or 'Data Scientist'
    match = re.search(r'(?:^|\n)([A-Za-z\s]+(?:Engineer|Developer|Scientist|Manager|Analyst|Architect))', cleaned_text, re.I)
    if match: 
        title = match.group(1).strip().title()
    
    # 3. Extract Top Achievements from cleaned text
    # We re-process the cleaned text to ensure achievements don't contain names
    doc = nlp(cleaned_text)
    sentences = [s.text.strip() for s in doc.sents if len(s.text.split()) > 5]
    
    scored = []
    for s in sentences:
        # Calculate a relevance score based on your KEYWORD_MAP
        score = sum(1 for k_list in KEYWORD_MAP.values() for k in k_list if k.lower() in s.lower())
        scored.append((s, score))
    
    # Sort by score descending
    scored.sort(key=lambda x: x[1], reverse=True)
    
    # Return the title and the top 3 high-scoring sentences
    return title, [s[0] for s in scored[:3]]

In [8]:
def generate_llm_summary(role, title, achievements):
    _, model, tokenizer = load_resources()
    
    prompt = f"Write a professional 4-line summary for a {title} who is a {role}. " \
             f"Highlight these achievements: {'; '.join(achievements)}"
    
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
    if torch.cuda.is_available(): inputs = {k: v.to("cuda") for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.7, do_sample=True)
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

def run_full_analysis(resume_file):
    with open(resume_file, 'r', encoding='utf-8') as f:
        text = f.read()
    
    # Process
    #scores = score_resume(text)
    nlp, _, _ = load_resources()
    title, achievements = extract_metadata(text, nlp)
    summary = generate_llm_summary(dominant, title, achievements)
    
    # Output
    print(f"\nGenerated Summary:\n{'-'*50}\n{summary}\n{'-'*50}")

In [9]:
run_full_analysis('resume.txt')


Generated Summary:
--------------------------------------------------
Founded and managed a team of engineers and developers to leverage AWS to create machine learning models, analytics, and automation solutions.
--------------------------------------------------
